# WiDS 2026 | Ultra Survival Stack + Near-Zone Expert Blend

In [1]:

import os
import sys
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

def _ensure(pkg: str, import_name: str | None = None) -> None:
    name = import_name or pkg
    try:
        __import__(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("scikit-survival", "sksurv")
_ensure("lightgbm", "lightgbm")
_ensure("scikit-learn", "sklearn")

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier

from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv


# ======================================================================================
# Config
# ======================================================================================
STRAT_THR = 5000.0
POWER_CAL_24 = 1.10
HORIZONS_PRED = np.array([12.0, 24.0, 48.0, 72.0], dtype=float)

GBSA_SEEDS = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
)
COX_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033)
RSF_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033)
LGB_NEAR_SEEDS = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
)
LGB_FAR_SEEDS = (
    8141, 7752, 432, 906, 6217,
    7785, 1603, 7609, 965, 2506,
    3771, 7080, 4963, 7939, 2751,
    473, 339, 3675, 5535, 4760,
)
NEAR_DIRECT_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033)

# sample3 / sample4 weights
W_GBSA_NEAR_12 = 0.76;  W_COX_NEAR_12 = 0.12;  W_RSF_NEAR_12 = 0.02;  W_LGB_NEAR_12 = 0.10
W_GBSA_NEAR_24 = 0.82;  W_COX_NEAR_24 = 0.14;  W_RSF_NEAR_24 = 0.02;  W_LGB_NEAR_24 = 0.02
W_GBSA_NEAR_48 = 0.73;  W_COX_NEAR_48 = 0.16;  W_RSF_NEAR_48 = 0.03;  W_LGB_NEAR_48 = 0.08

W_GBSA_FAR_24 = 0.62;  W_COX_FAR_24 = 0.25;  W_RSF_FAR_24 = 0.06;  W_LGB_FAR_24 = 0.07
W_GBSA_FAR_48 = 0.35;  W_COX_FAR_48 = 0.22;  W_RSF_FAR_48 = 0.06;  W_LGB_FAR_48 = 0.37

W_GBSA_NEAR_12_ALT = 0.76;  W_COX_NEAR_12_ALT = 0.12;  W_LGB_NEAR_12_ALT = 0.10;  W_REM_NEAR_12_ALT = 0.02
W_GBSA_NEAR_24_ALT = 0.82;  W_COX_NEAR_24_ALT = 0.14;  W_LGB_NEAR_24_ALT = 0.02;  W_REM_NEAR_24_ALT = 0.02
W_GBSA_NEAR_48_ALT = 0.73;  W_COX_NEAR_48_ALT = 0.16;  W_LGB_NEAR_48_ALT = 0.08;  W_REM_NEAR_48_ALT = 0.03

W_GBSA_FAR_24_ALT = 0.62;  W_COX_FAR_24_ALT = 0.25;  W_LGB_FAR_24_ALT = 0.07;  W_REM_FAR_24_ALT = 0.06
W_GBSA_FAR_48_ALT = 0.35;  W_COX_FAR_48_ALT = 0.22;  W_LGB_FAR_48_ALT = 0.37;  W_REM_FAR_48_ALT = 0.06


# ======================================================================================
# Utilities
# ======================================================================================
def locate_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26"),
        Path("/kaggle/input/widsworldwide-globaldathon26"),
        Path("/kaggle/input/WiDSWorldWide_GlobalDathon26"),
        Path("../input/competitions/WiDSWorldWide_GlobalDathon26"),
        Path("../input/widsworldwide-globaldathon26"),
    ]
    for p in candidates:
        if (p / "train.csv").exists() and (p / "test.csv").exists() and (p / "sample_submission.csv").exists():
            return p

    root = Path("/kaggle/input")
    if root.exists():
        for p in root.rglob("train.csv"):
            parent = p.parent
            if (parent / "test.csv").exists() and (parent / "sample_submission.csv").exists():
                return parent

    fallback = Path(".")
    if (fallback / "train.csv").exists() and (fallback / "test.csv").exists() and (fallback / "sample_submission.csv").exists():
        return fallback

    raise FileNotFoundError("Could not find train.csv / test.csv / sample_submission.csv")


def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def rank_to_unit(x: np.ndarray) -> np.ndarray:
    s = pd.Series(np.asarray(x).ravel())
    return ((s.rank(method="average").to_numpy() - 0.5) / len(s)).reshape(-1)


def enforce_monotonicity(preds: np.ndarray) -> np.ndarray:
    out = np.clip(preds, 0.0, 1.0)
    for j in range(1, out.shape[1]):
        out[:, j] = np.maximum(out[:, j], out[:, j - 1])
    return out


def get_surv_predictions(model, X: pd.DataFrame, horizons: np.ndarray = HORIZONS_PRED) -> np.ndarray:
    surv_fns = model.predict_survival_function(X)
    out = np.empty((len(surv_fns), len(horizons)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        out[i, :] = fn(np.clip(horizons, t_min, t_max))
    return 1.0 - out


def make_binary_target(time_vals: np.ndarray, event_vals: np.ndarray, horizon: float) -> tuple[np.ndarray, np.ndarray]:
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown


def compute_ipcw_weights(times: np.ndarray, events: np.ndarray, horizon: float) -> np.ndarray:
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)

    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        censored_at_t = ((times == t) & (events == 0)).sum()
        step = 1.0 - (censored_at_t / at_risk) if at_risk > 0 else 1.0
        surv[i] = step if i == 0 else surv[i - 1] * step

    def G(t: float) -> float:
        idx = np.searchsorted(unique_t, t, side="right") - 1
        return max(float(surv[idx]), 0.01) if idx >= 0 else 1.0

    w = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            w[i] = 1.0 / G(float(times[i]))
        elif times[i] >= horizon:
            w[i] = 1.0 / G(float(horizon))
        else:
            w[i] = 1.0
    return w


def near_soft_gate(dist_m: np.ndarray, thr: float = STRAT_THR, decay: float = 350.0) -> np.ndarray:
    dist_m = np.asarray(dist_m, dtype=float)
    out = np.ones_like(dist_m)
    mask = dist_m > thr
    out[mask] = np.exp(-(dist_m[mask] - thr) / decay)
    return np.clip(out, 0.0, 1.0)


# ======================================================================================
# Features
# ======================================================================================
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dist = out["dist_min_ci_0_5h"].clip(lower=1.0)
    speed = out["closing_speed_m_per_h"]
    area_first = out["area_first_ha"]
    per = out["num_perimeters_0_5h"]

    out["log_distance"] = np.log1p(dist)
    out["inv_distance"] = 1.0 / (dist / 1000.0 + 0.1)
    out["inv_distance_sq"] = out["inv_distance"] ** 2
    out["sqrt_distance"] = np.sqrt(dist)
    out["dist_km"] = dist / 1000.0
    out["dist_km_sq"] = out["dist_km"] ** 2
    out["dist_rank"] = dist.rank(pct=True)
    out["log_area"] = np.log1p(area_first)

    fire_radius = np.sqrt(area_first * 10000.0 / np.pi)
    out["fire_radius_km"] = fire_radius / 1000.0
    out["radius_to_dist"] = fire_radius / dist
    out["area_to_dist_ratio"] = area_first / (dist / 1000.0 + 0.1)
    out["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)
    out["dist_x_log_area"] = out["dist_km"] * out["log_area"]

    closing_pos = speed.clip(lower=0.0)
    radial_growth = out["radial_growth_rate_m_per_h"].clip(lower=0.0)
    effective_closing = closing_pos + radial_growth

    out["has_movement"] = (per > 1).astype(float)
    out["single_perimeter"] = (per <= 1).astype(float)
    out["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999.0)
    out["log_eta"] = np.log1p(out["eta_hours"].clip(0.0, 9999.0))
    out["effective_closing_speed"] = effective_closing
    out["eta_effective"] = np.where(effective_closing > 0.01, dist / effective_closing, 9999.0).clip(max=9999.0)

    out["threat_score"] = out["alignment_abs"] * speed / np.log1p(dist)
    out["fire_urgency"] = per * speed
    out["growth_intensity"] = out["area_growth_rate_ha_per_h"] * per
    out["movement_score"] = closing_pos + radial_growth + np.maximum(out["area_growth_rate_ha_per_h"], 0.0)
    out["speed_x_align"] = closing_pos * out["alignment_abs"]
    out["resolution_score"] = per * np.maximum(out["dt_first_last_0_5h"], 0.0)

    out["zone_near"] = (dist < STRAT_THR).astype(float)
    out["zone_far"] = (dist >= STRAT_THR).astype(float)
    out["is_summer"] = out["event_start_month"].isin([6, 7, 8]).astype(float)
    out["is_afternoon"] = ((out["event_start_hour"] >= 12) & (out["event_start_hour"] < 20)).astype(float)

    zero_speed = (np.abs(out["closing_speed_m_per_h"]) < 1e-12).astype(float)
    zero_radial = (np.abs(out["radial_growth_rate_m_per_h"]) < 1e-12).astype(float)
    zero_growth = (np.abs(out["area_growth_rate_ha_per_h"]) < 1e-12).astype(float)
    out["zero_speed"] = zero_speed
    out["zero_radial"] = zero_radial
    out["zero_growth"] = zero_growth
    out["static_fire"] = ((per <= 1) & (zero_speed > 0) & (zero_radial > 0) & (zero_growth > 0)).astype(float)
    out["slow_case"] = ((out["low_temporal_resolution_0_5h"] > 0) | (per <= 1)).astype(float)

    near_mask = dist < STRAT_THR
    out["near_speed_rank"] = 0.0
    if near_mask.sum() > 0:
        out.loc[near_mask, "near_speed_rank"] = speed[near_mask].rank(pct=True)
    far_mask = ~near_mask
    out["far_threat_rank"] = 0.0
    if far_mask.sum() > 0:
        out.loc[far_mask, "far_threat_rank"] = out.loc[far_mask, "threat_score"].rank(pct=True)

    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out


# ======================================================================================
# Model blocks
# ======================================================================================
def fit_gbsa_family(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    X_train = train_df.drop(columns=["event_id", "event", "time_to_hit_hours"])
    X_test = test_df.drop(columns=["event_id"])
    y_surv = Surv.from_arrays(event=train_df["event"].astype(bool), time=train_df["time_to_hit_hours"])

    gbsa_configs = [
        {"learning_rate": 0.01,  "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
        {"learning_rate": 0.01,  "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 15, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
        {"learning_rate": 0.01,  "subsample": 0.60, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
        {"learning_rate": 0.005, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 2000, "dropout_rate": 0.0},
        {"learning_rate": 0.01,  "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 3, "n_estimators": 1400, "dropout_rate": 0.0},
        {"learning_rate": 0.008, "subsample": 0.75, "max_depth": 2, "min_samples_leaf": 15, "min_samples_split": 4, "n_estimators": 1500, "dropout_rate": 0.0},
        {"learning_rate": 0.015, "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 1000, "dropout_rate": 0.0},
        {"learning_rate": 0.005, "subsample": 0.90, "max_depth": 3, "min_samples_leaf": 18, "min_samples_split": 5, "n_estimators": 2500, "dropout_rate": 0.0},
        {"learning_rate": 0.01,  "subsample": 0.80, "max_depth": 4, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200, "dropout_rate": 0.0},
        {"learning_rate": 0.02,  "subsample": 0.65, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 800,  "dropout_rate": 0.0},
    ]

    out = np.zeros((len(X_test), 4), dtype=float)
    event_values = train_df["event"].to_numpy()

    print(f"GBSA: {len(gbsa_configs)} configs x {len(GBSA_SEEDS)} seeds x 5-fold")
    for cfg_idx, cfg in enumerate(gbsa_configs, start=1):
        cfg_pred = np.zeros_like(out)
        for seed in GBSA_SEEDS:
            seed_pred = np.zeros_like(out)
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr_idx, _ in cv.split(X_train, event_values):
                m = GradientBoostingSurvivalAnalysis(**{**cfg, "random_state": seed})
                m.fit(X_train.iloc[tr_idx], y_surv[tr_idx])
                seed_pred += get_surv_predictions(m, X_test) / 5.0
            cfg_pred += seed_pred / len(GBSA_SEEDS)
        out += cfg_pred / len(gbsa_configs)
        print(f"  GBSA cfg {cfg_idx}/{len(gbsa_configs)} done")

    out[:, 1] = np.clip(out[:, 1] ** POWER_CAL_24, 0.0, 1.0)
    return np.clip(out, 0.0, 1.0)


def fit_cox_family(train_processed: pd.DataFrame, test_processed: pd.DataFrame, event_values: np.ndarray, y_surv) -> np.ndarray:
    features = [
        "dist_km", "log_distance", "inv_distance",
        "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
        "alignment_abs", "threat_score", "log_eta", "eta_effective",
        "area_to_dist_ratio", "fire_radius_km",
        "num_perimeters_0_5h", "has_movement",
        "near_speed_rank", "far_threat_rank",
        "is_summer", "is_afternoon",
        "zone_near", "zone_far",
        "slow_case", "static_fire", "resolution_score", "dist_x_log_area",
    ]
    use = [f for f in features if f in train_processed.columns]
    X_train = train_processed[use].copy()
    X_test = test_processed[use].copy()

    scaler = StandardScaler()
    X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=use, index=X_train.index)
    X_test_sc = pd.DataFrame(scaler.transform(X_test), columns=use, index=X_test.index)

    alphas = [0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00]
    out = np.zeros((len(X_test_sc), 4), dtype=float)

    print(f"CoxPH: {len(alphas)} alphas x {len(COX_SEEDS)} seeds x 5-fold")
    for alpha in alphas:
        alpha_pred = np.zeros_like(out)
        for seed in COX_SEEDS:
            seed_pred = np.zeros_like(out)
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr_idx, _ in cv.split(X_train_sc, event_values):
                try:
                    m = CoxPHSurvivalAnalysis(alpha=alpha)
                    m.fit(X_train_sc.iloc[tr_idx], y_surv[tr_idx])
                    seed_pred += get_surv_predictions(m, X_test_sc) / 5.0
                except Exception:
                    seed_pred += 0.5 / 5.0
            alpha_pred += seed_pred / len(COX_SEEDS)
        out += alpha_pred / len(alphas)
        print(f"  Cox alpha={alpha} done")

    return np.clip(out, 0.0, 1.0)


def fit_rsf_family(train_df: pd.DataFrame, test_df: pd.DataFrame, event_values: np.ndarray, y_surv) -> np.ndarray:
    X_train = train_df.drop(columns=["event_id", "event", "time_to_hit_hours"])
    X_test = test_df.drop(columns=["event_id"])

    rsf_configs = [
        {"n_estimators": 200, "min_samples_leaf": 12, "max_features": "sqrt", "max_depth": None},
        {"n_estimators": 200, "min_samples_leaf": 18, "max_features": "sqrt", "max_depth": None},
        {"n_estimators": 200, "min_samples_leaf": 12, "max_features": 0.5,    "max_depth": 5},
    ]

    out = np.zeros((len(X_test), 4), dtype=float)

    print(f"RSF: {len(rsf_configs)} configs x {len(RSF_SEEDS)} seeds x 5-fold")
    for cfg_idx, cfg in enumerate(rsf_configs, start=1):
        cfg_pred = np.zeros_like(out)
        for seed in RSF_SEEDS:
            seed_pred = np.zeros_like(out)
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr_idx, _ in cv.split(X_train, event_values):
                m = RandomSurvivalForest(**{**cfg, "random_state": seed, "n_jobs": -1})
                m.fit(X_train.iloc[tr_idx], y_surv[tr_idx])
                seed_pred += get_surv_predictions(m, X_test) / 5.0
            cfg_pred += seed_pred / len(RSF_SEEDS)
        out += cfg_pred / len(rsf_configs)
        print(f"  RSF cfg {cfg_idx}/{len(rsf_configs)} done")

    return np.clip(out, 0.0, 1.0)


def fit_zone_split_lgb(
    train_processed: pd.DataFrame,
    test_processed: pd.DataFrame,
    time_values: np.ndarray,
    event_values: np.ndarray,
) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray]]:
    near_features = [
        "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
        "alignment_abs", "num_perimeters_0_5h", "area_growth_rate_ha_per_h",
        "eta_effective", "log_eta", "dist_km", "threat_score",
        "near_speed_rank", "event_start_hour", "is_afternoon", "fire_urgency",
        "area_first_ha", "fire_radius_km",
        "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
        "slow_case", "static_fire", "resolution_score", "movement_score",
    ]
    far_features = [
        "dist_km", "log_distance", "inv_distance",
        "closing_speed_m_per_h", "alignment_abs",
        "threat_score", "log_eta", "eta_effective",
        "area_to_dist_ratio", "num_perimeters_0_5h",
        "far_threat_rank", "is_summer", "zone_far",
        "slow_case", "dist_x_log_area",
    ]

    use_near = [f for f in near_features if f in train_processed.columns]
    use_far = [f for f in far_features if f in train_processed.columns]

    X_near_train = train_processed[use_near].copy()
    X_near_test = test_processed[use_near].copy()
    X_far_train = train_processed[use_far].copy()
    X_far_test = test_processed[use_far].copy()

    near_cfgs = {
        24: {"max_depth": 2, "learning_rate": 0.04, "n_estimators": 200, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_samples": 4, "reg_alpha": 0.5, "reg_lambda": 1.5, "num_leaves": 4},
        48: {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 150, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_samples": 3, "reg_alpha": 0.3, "reg_lambda": 1.0, "num_leaves": 4},
    }
    far_cfgs = {
        24: {"max_depth": 2, "learning_rate": 0.03, "n_estimators": 200, "subsample": 0.7, "colsample_bytree": 0.7, "min_child_samples": 8, "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 4},
        48: {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 150, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_samples": 6, "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 4},
    }

    out_near: dict[int, np.ndarray] = {}
    out_far: dict[int, np.ndarray] = {}

    print(f"Zone-split LGB: {len(LGB_NEAR_SEEDS)} near seeds | {len(LGB_FAR_SEEDS)} far seeds")
    for horizon in [24, 48]:
        y_bin, mask = make_binary_target(time_values, event_values, horizon)
        valid_idx = np.where(mask)[0]
        w_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)

        pred_near = np.zeros(len(X_near_test), dtype=float)
        for seed in LGB_NEAR_SEEDS:
            clf = lgb.LGBMClassifier(
                **near_cfgs[horizon],
                objective="binary",
                random_state=seed,
                verbose=-1,
                class_weight="balanced",
                n_jobs=-1,
            )
            clf.fit(X_near_train.iloc[valid_idx], y_bin[valid_idx], sample_weight=w_full)
            pred_near += clf.predict_proba(X_near_test)[:, 1] / len(LGB_NEAR_SEEDS)
        out_near[horizon] = np.clip(pred_near, 0.0, 1.0)

        pred_far = np.zeros(len(X_far_test), dtype=float)
        for seed in LGB_FAR_SEEDS:
            clf = lgb.LGBMClassifier(
                **far_cfgs[horizon],
                objective="binary",
                random_state=seed,
                verbose=-1,
                class_weight="balanced",
                n_jobs=-1,
            )
            clf.fit(X_far_train.iloc[valid_idx], y_bin[valid_idx], sample_weight=w_full)
            pred_far += clf.predict_proba(X_far_test)[:, 1] / len(LGB_FAR_SEEDS)
        out_far[horizon] = np.clip(pred_far, 0.0, 1.0)

        print(f"  LGB horizon {horizon}h done")

    return out_near, out_far


def fit_near_direct_models(train_processed: pd.DataFrame, test_processed: pd.DataFrame, train_df: pd.DataFrame) -> np.ndarray:
    near_train_mask = train_df["dist_min_ci_0_5h"].to_numpy() < STRAT_THR
    dist_test = test_processed["dist_min_ci_0_5h"].to_numpy()
    gate = near_soft_gate(dist_test)

    features = [
        "low_temporal_resolution_0_5h", "num_perimeters_0_5h", "dt_first_last_0_5h",
        "log_area", "area_first_ha", "fire_radius_km",
        "dist_km", "log_distance", "dist_x_log_area",
        "event_start_hour", "event_start_month",
        "threat_score", "alignment_abs",
        "closing_speed_m_per_h", "radial_growth_rate_m_per_h", "area_growth_rate_ha_per_h",
        "eta_effective", "near_speed_rank",
        "single_perimeter", "slow_case", "static_fire", "movement_score", "resolution_score",
    ]
    use = [f for f in features if f in train_processed.columns]

    X_train_near = train_processed.loc[near_train_mask, use].copy().reset_index(drop=True)
    X_test_all = test_processed[use].copy()
    time_near = train_df.loc[near_train_mask, "time_to_hit_hours"].to_numpy()

    lgb_cfgs = {
        12: {"max_depth": 2, "learning_rate": 0.04, "n_estimators": 200, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_samples": 3, "reg_alpha": 0.5, "reg_lambda": 1.0, "num_leaves": 4},
        24: {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 180, "subsample": 0.85, "colsample_bytree": 0.8, "min_child_samples": 3, "reg_alpha": 0.5, "reg_lambda": 1.2, "num_leaves": 4},
        48: {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 160, "subsample": 0.90, "colsample_bytree": 0.85, "min_child_samples": 2, "reg_alpha": 0.3, "reg_lambda": 1.0, "num_leaves": 4},
    }

    out = np.zeros((len(X_test_all), 3), dtype=float)

    print(f"Near-direct horizons: {len(NEAR_DIRECT_SEEDS)} seeds")
    for j, horizon in enumerate([12, 24, 48]):
        y = (time_near <= horizon).astype(int)
        base_rate = float(y.mean())
        pred = np.zeros(len(X_test_all), dtype=float)

        for seed in NEAR_DIRECT_SEEDS:
            clf_lgb = lgb.LGBMClassifier(
                **lgb_cfgs[horizon],
                objective="binary",
                random_state=seed,
                class_weight="balanced",
                verbose=-1,
                n_jobs=-1,
            )
            clf_lgb.fit(X_train_near, y)
            p_lgb = clf_lgb.predict_proba(X_test_all)[:, 1]

            clf_logit = Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(C=0.5, max_iter=400, class_weight="balanced", random_state=seed)),
            ])
            clf_logit.fit(X_train_near, y)
            p_logit = clf_logit.predict_proba(X_test_all)[:, 1]

            clf_et = ExtraTreesClassifier(
                n_estimators=300,
                max_depth=4,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=seed,
                n_jobs=-1,
            )
            clf_et.fit(X_train_near, y)
            p_et = clf_et.predict_proba(X_test_all)[:, 1]

            pred += (0.50 * p_lgb + 0.30 * p_logit + 0.20 * p_et) / len(NEAR_DIRECT_SEEDS)

        pred = 0.85 * pred + 0.15 * base_rate
        pred = gate * pred
        out[:, j] = np.clip(pred, 0.0, 1.0)

        print(f"  Near-direct {horizon}h done")

    return out


def fit_rule_model(train_processed: pd.DataFrame, test_processed: pd.DataFrame, train_df: pd.DataFrame) -> np.ndarray:
    near_train_mask = train_df["dist_min_ci_0_5h"].to_numpy() < STRAT_THR
    near_train = train_processed.loc[near_train_mask].copy().reset_index(drop=True)
    test_all = test_processed.copy()

    def nper_bucket(x: pd.Series) -> pd.Series:
        return np.where(x <= 1, 1, np.where(x == 2, 2, 3))

    train_nper = nper_bucket(near_train["num_perimeters_0_5h"])
    test_nper = nper_bucket(test_all["num_perimeters_0_5h"])

    q_edges = np.unique(np.quantile(near_train["log_area"], [0.0, 0.25, 0.5, 0.75, 1.0]))
    if len(q_edges) < 3:
        q_edges = np.array([near_train["log_area"].min(), near_train["log_area"].median(), near_train["log_area"].max() + 1e-6])
    q_edges[0] -= 1e-9
    q_edges[-1] += 1e-9

    train_area_bin = pd.cut(near_train["log_area"], bins=q_edges, labels=False, include_lowest=True).astype(int)
    test_area_bin = pd.cut(test_all["log_area"], bins=q_edges, labels=False, include_lowest=True).fillna(0).astype(int)

    gate = near_soft_gate(test_all["dist_min_ci_0_5h"].to_numpy())
    out = np.zeros((len(test_all), 3), dtype=float)

    for j, horizon in enumerate([12, 24, 48]):
        y = (train_df.loc[near_train_mask, "time_to_hit_hours"].to_numpy() <= horizon).astype(float)

        global_mean = float(y.mean())

        tmp = pd.DataFrame({
            "low": near_train["low_temporal_resolution_0_5h"].astype(int).to_numpy(),
            "nper": train_nper.astype(int),
            "abin": train_area_bin.astype(int),
            "y": y,
        })

        g1 = tmp.groupby(["low"])["y"].agg(["sum", "count"]).to_dict("index")
        g2 = tmp.groupby(["low", "nper"])["y"].agg(["sum", "count"]).to_dict("index")
        g3 = tmp.groupby(["low", "nper", "abin"])["y"].agg(["sum", "count"]).to_dict("index")

        pred = np.zeros(len(test_all), dtype=float)
        low_test = test_all["low_temporal_resolution_0_5h"].astype(int).to_numpy()

        for i in range(len(test_all)):
            low = int(low_test[i])
            nb = int(test_nper[i])
            ab = int(test_area_bin.iloc[i])

            parent1 = global_mean
            if low in g1:
                parent1 = (g1[low]["sum"] + 3.0 * global_mean) / (g1[low]["count"] + 3.0)

            parent2 = parent1
            key2 = (low, nb)
            if key2 in g2:
                parent2 = (g2[key2]["sum"] + 2.0 * parent1) / (g2[key2]["count"] + 2.0)

            parent3 = parent2
            key3 = (low, nb, ab)
            if key3 in g3:
                parent3 = (g3[key3]["sum"] + 1.5 * parent2) / (g3[key3]["count"] + 1.5)

            pred[i] = parent3

        out[:, j] = np.clip(gate * pred, 0.0, 1.0)

    return out


def blend_zone_core(
    gbsa: np.ndarray,
    cox: np.ndarray,
    rsf: np.ndarray,
    lgb_near: np.ndarray,
    lgb_far: np.ndarray,
    near_mask: np.ndarray,
) -> np.ndarray:
    out = np.zeros((len(gbsa), 4), dtype=float)

    out[:, 0] = np.where(
        near_mask,
        W_GBSA_NEAR_12 * gbsa[:, 0] + W_COX_NEAR_12 * cox[:, 0] + W_RSF_NEAR_12 * rsf[:, 0] + W_LGB_NEAR_12 * lgb_near[24],
        gbsa[:, 0],
    )
    out[:, 1] = np.where(
        near_mask,
        W_GBSA_NEAR_24 * gbsa[:, 1] + W_COX_NEAR_24 * cox[:, 1] + W_RSF_NEAR_24 * rsf[:, 1] + W_LGB_NEAR_24 * lgb_near[24],
        W_GBSA_FAR_24 * gbsa[:, 1] + W_COX_FAR_24 * cox[:, 1] + W_RSF_FAR_24 * rsf[:, 1] + W_LGB_FAR_24 * lgb_far[24],
    )
    out[:, 2] = np.where(
        near_mask,
        W_GBSA_NEAR_48 * gbsa[:, 2] + W_COX_NEAR_48 * cox[:, 2] + W_RSF_NEAR_48 * rsf[:, 2] + W_LGB_NEAR_48 * lgb_near[48],
        W_GBSA_FAR_48 * gbsa[:, 2] + W_COX_FAR_48 * cox[:, 2] + W_RSF_FAR_48 * rsf[:, 2] + W_LGB_FAR_48 * lgb_far[48],
    )
    out[:, 3] = 1.0
    return np.clip(out, 0.0, 1.0)


def blend_zone_alt(
    gbsa: np.ndarray,
    cox: np.ndarray,
    lgb_near: np.ndarray,
    lgb_far: np.ndarray,
    near_mask: np.ndarray,
) -> np.ndarray:
    out = np.zeros((len(gbsa), 4), dtype=float)

    out[:, 0] = np.where(
        near_mask,
        W_GBSA_NEAR_12_ALT * gbsa[:, 0] + W_COX_NEAR_12_ALT * cox[:, 0] + W_LGB_NEAR_12_ALT * lgb_near[24] + W_REM_NEAR_12_ALT * 0.5,
        gbsa[:, 0],
    )
    out[:, 1] = np.where(
        near_mask,
        W_GBSA_NEAR_24_ALT * gbsa[:, 1] + W_COX_NEAR_24_ALT * cox[:, 1] + W_LGB_NEAR_24_ALT * lgb_near[24] + W_REM_NEAR_24_ALT * 0.5,
        W_GBSA_FAR_24_ALT * gbsa[:, 1] + W_COX_FAR_24_ALT * cox[:, 1] + W_LGB_FAR_24_ALT * lgb_far[24] + W_REM_FAR_24_ALT * 0.5,
    )
    out[:, 2] = np.where(
        near_mask,
        W_GBSA_NEAR_48_ALT * gbsa[:, 2] + W_COX_NEAR_48_ALT * cox[:, 2] + W_LGB_NEAR_48_ALT * lgb_near[48] + W_REM_NEAR_48_ALT * 0.5,
        W_GBSA_FAR_48_ALT * gbsa[:, 2] + W_COX_FAR_48_ALT * cox[:, 2] + W_LGB_FAR_48_ALT * lgb_far[48] + W_REM_FAR_48_ALT * 0.5,
    )
    out[:, 3] = 1.0
    return np.clip(out, 0.0, 1.0)


# ======================================================================================
# Main
# ======================================================================================
data_dir = locate_data_dir()
output_path = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")

train_df = pd.read_csv(data_dir / "train.csv")
test_df = pd.read_csv(data_dir / "test.csv")
sample_sub = pd.read_csv(data_dir / "sample_submission.csv")

train_processed = create_features(train_df)
test_processed = create_features(test_df)

event_values = train_df["event"].to_numpy()
time_values = train_df["time_to_hit_hours"].to_numpy()
dist_test = test_df["dist_min_ci_0_5h"].to_numpy()
near_test = dist_test < STRAT_THR
y_surv = Surv.from_arrays(event=train_df["event"].astype(bool), time=train_df["time_to_hit_hours"])

print(f"DATA_DIR     : {data_dir}")
print(f"TRAIN / TEST : {train_df.shape} / {test_df.shape}")
print(f"NEAR TEST    : {int(near_test.sum())}")

test_gbsa = fit_gbsa_family(train_df, test_df)
test_cox = fit_cox_family(train_processed, test_processed, event_values, y_surv)
test_rsf = fit_rsf_family(train_df, test_df, event_values, y_surv)
lgb_near_test, lgb_far_test = fit_zone_split_lgb(train_processed, test_processed, time_values, event_values)
test_near_direct = fit_near_direct_models(train_processed, test_processed, train_df)
test_rule = fit_rule_model(train_processed, test_processed, train_df)

core = blend_zone_core(test_gbsa, test_cox, test_rsf, lgb_near_test, lgb_far_test, near_test)
alt = blend_zone_alt(test_gbsa, test_cox, lgb_near_test, lgb_far_test, near_test)

final = np.zeros_like(core)
final[:, 1] = (
    0.60 * core[:, 1]
    + 0.10 * alt[:, 1]
    + 0.20 * test_near_direct[:, 1]
    + 0.10 * test_rule[:, 1]
)
final[:, 2] = (
    0.65 * core[:, 2]
    + 0.10 * alt[:, 2]
    + 0.10 * test_near_direct[:, 2]
    + 0.15 * test_rule[:, 2]
)
p12_prob = (
    0.45 * core[:, 0]
    + 0.10 * alt[:, 0]
    + 0.25 * test_near_direct[:, 0]
    + 0.20 * test_rule[:, 0]
)
p12_rank = (
    rank_to_unit(0.50 * core[:, 0] + 0.30 * test_near_direct[:, 0] + 0.20 * test_rule[:, 0])
)
p12 = 0.75 * p12_prob + 0.25 * p12_rank
p12 = np.minimum(p12, np.maximum(final[:, 1] - 1e-6, 0.0))
final[:, 0] = np.clip(p12, 0.0, 1.0)
final[:, 3] = 1.0

final = enforce_monotonicity(final)

submission = pd.DataFrame(
    {
        "event_id": test_df["event_id"].to_numpy(),
        "prob_12h": final[:, 0],
        "prob_24h": final[:, 1],
        "prob_48h": final[:, 2],
        "prob_72h": final[:, 3],
    }
)
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left")

assert submission["event_id"].is_unique
assert submission["event_id"].tolist() == sample_sub["event_id"].tolist()
assert ((submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] >= 0.0) & (submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] <= 1.0)).all().all()
assert (submission["prob_12h"] <= submission["prob_24h"]).all()
assert (submission["prob_24h"] <= submission["prob_48h"]).all()
assert (submission["prob_48h"] <= submission["prob_72h"]).all()

submission.to_csv(output_path, index=False)

print(f"SAVED        : {output_path}")
print(submission.head(10).to_string(index=False))
print(submission.describe().round(6))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 7.7 MB/s eta 0:00:00
DATA_DIR     : /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN / TEST : (221, 37) / (95, 35)
NEAR TEST    : 28
GBSA: 10 configs x 20 seeds x 5-fold
  GBSA cfg 1/10 done
  GBSA cfg 2/10 done
  GBSA cfg 3/10 done
  GBSA cfg 4/10 done
  GBSA cfg 5/10 done
  GBSA cfg 6/10 done
  GBSA cfg 7/10 done
  GBSA cfg 8/10 done
  GBSA cfg 9/10 done
  GBSA cfg 10/10 done
CoxPH: 7 alphas x 10 seeds x 5-fold
  Cox alpha=0.001 done
  Cox alpha=0.01 done
  Cox alpha=0.05 done
  Cox alpha=0.1 done
  Cox alpha=0.5 done
  Cox alpha=1.0 done
  Cox alpha=2.0 done
RSF: 3 configs x 10 seeds x 5-fold
  RSF cfg 1/3 done
  RSF cfg 2/3 done
  RSF cfg 3/3 done
Zone-split LGB: 20 near seeds | 20 far seeds
  LGB horizon 24h done
  LGB horizon 48h done
Near-direct horizons: 